# Inverse Problems 2026 - Exercise 4 (home exercises)

Return by **Fri 27.11.2026 at 23:59** in Moodle! You can ask for help with these exercises in the exercise session the same day (10:00, room 7334).

The returned exercises should be renamed `Exercise04_lastname_firstname.ipynb` and contain your solutions in clear and readable manner and format. The programming exercises need to be runnable with minimal effort. If you find markdown too cumbersome for writing the necessary math, you can attach **a single extra pdf-file** for the theoretical exercises and denote it `Exercise04_lastname_firstname.pdf`.

## Task 1 (Direct problem)
Recall the groundwater filtration example from the lecture slides: for
any permittivity $\sigma \in C^1[0,1]$ and source $g \in C[0,1]$, let $u \in C^2[0,1]$ be the
solution of
\begin{align}
-\frac{\partial}{\partial x} \left( \sigma \frac{\partial u}{\partial x} \right) (x) &= g(x) \text{ on } [0,1] \\
\frac{\partial u}{\partial x}(0) &= 0 \nonumber \\
u(1) &= 0 \nonumber
\end{align}
Discretize the equation for the variable $U = [u_0,...,u_N]^T \in \mathbb{R}^{N+1}$, where $u_0 = u_1$ and
$u_N = 0$ according to the boundary values and $\sigma \geq \sigma_0$ for some constant
$\sigma_0 > 0$.

Given some functions $g$ and $\sigma$, write a Python routine `solveU(sigma)` for the solution operator $F_g : \sigma \mapsto U$ of
the direct problem using finite difference approximations of the derivative. The following script is a good starting point.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as op

# set dimension and a lower limit to sigma
N = 101
sigma0 = 0.2

# contruct ground truth sigma using an exponential mollifier 
sigma_true = sigma0*np.ones((N,))
with np.errstate(divide='ignore'): # Silence division by zero errors
    sigma_true[(N//3):2*(N//3)] = np.exp(-1/(1 - np.linspace(-1,1, N//3, endpoint=True)**2)) + sigma0
xs = np.linspace(0, 1, N, endpoint=True)

# plot the ground truth sigma
plt.plot(xs, sigma_true)
plt.title("True sigma")
plt.show()

In [ ]:
def solveU(sigma):
    """Given function `sigma`, solve the linear system given by the finite difference approximation and the boundary conditions."""
    N = len(sigma)
    g = np.sin(np.linspace(0,np.pi,N-1, endpoint=True)).reshape((-1,1)) # Fixed g(x) := sin(pi*x)

    # TODO: finite difference matrices
    L1 = None # N x N+1 finite difference matrix
    L2 = None # N-1 x N finite difference matrix

    # TODO
    # Boundary condition: u_0 = u_1 <-> u_1 - u_0 = 0 = <U, b1>
    b1 = None

    # TODO
    # Boundary condition: u_N = 0 = <U, b2>
    b2 = None

    # Stacked form
    sigma = sigma.reshape((-1, 1)) # Column vector -> sigma * L1 : scale every row of L1 by sigma_i
    K = -L2 @ (sigma * L1)
    K = np.concatenate((K, b1, b2), axis=0)

    # TODO
    extra = None
    g0 = np.concatenate((g, extra), axis=0) # Add extra rows to data (due to the boundary conditions)

    u, *_ = np.linalg.lstsq(K, g0, rcond=None) # Solve K*u = g0
    return u.reshape((-1, 1)) # Return column matrix

# solve u (the data) and add white noise with standard deviation 0.01
u = solveU(sigma_true)

plt.plot(u, label="Clean data")

# TODO: add noise
delta = 0.01
un = u 

plt.plot(un, label="Noisy data")
plt.legend()
plt.show()

___
## Task 2 (Tikhonov regularization)
Using the routine produced in previous exercise, reconstruct the Tikhonov
regularized solution
\begin{equation*}
\sigma^\delta_\alpha := \argmin_{\sigma} \lbrace \| F_g(\sigma) − y^\delta \|^2 + \alpha \|\sigma − \sigma_0 \|^2 \rbrace
\end{equation*}
for a given regularization parameter $\alpha$ using gradient descent algorithm [`fmin_cg`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fmin_cg.html) (conjugate gradient) from `scipy.optimize`. Parameters and data are provided in the following Python script.

In [ ]:
from numpy.linalg import norm
# TODO: write objective function to be minimized
alpha = 0.05

def objFun(args):
    """Tikhonov regularized objetive function wrt. sigma"""
    return alpha*None

x0 = 0.01 * np.ones((N, 1)) # initial guess for sigma

# minimize it (this may take few minutes)
# Good results may require many more interations!
recon_cg = op.fmin_cg(objFun, x0, maxiter=200)

___
## Task 3 (Numerical Jacobian)
`op.fmin_cg` (and many other methods) in Scipy approximates the gradient (or Jacobian) numerically using finite difference if it is not provided explicitly. In the exercise session we will use the Jacobian to implement the Gauss-Newton method, which converges much faster.

Using forward finite differences
$$ \frac{\partial F}{\partial \sigma_i}(\sigma) \approx \frac{F(\sigma + h e_i) - F(\sigma)}{h}, $$
write a routine that estimates the Jacobian matrix $J_F(\sigma) \in \mathbb{R}^{N+1 \times N}$ (sometimes denoted $F'(\sigma)$). The following code is a good starting point. Test your routine on a simple function whose Jacobian you know, e.g. a linear map $F(x) = Bx$.

In [ ]:
def num_jacobian(x, F, h = 1e-8):
    """Finite difference approximation of the Jacobian matrix of operator `F`."""
    Fx = F(x)
    m = len(Fx)
    n = len(x)
    jac = np.zeros((m, n))

    # TODO: perturb components of x by h to get partial derivatives
    # Recall that i'th column of J are the partial derivatives wrt. x_i


    return jac / h

___
## Task 4 (Non-uniqueness)
Consider again the problem (1) and construct a counter-example to the inverse
problem, when $u \notin C_{\lambda,M}$ for all $\gamma \in \mathbb{R}^+$, such that it does not have a unique solution.

Consider the water density
\begin{equation}
u(x) = \begin{cases}
\exp \left( \frac{-1}{1 - (4x - 2)^2} \right), \ & x \in (\tfrac{1}{4}, \tfrac{3}{4} ) \\
0, & x  \in [0, \tfrac{1}{4}] \cup [ \tfrac{3}{4}, 1]
\end{cases}
\end{equation}
and define the source term as
\begin{equation*}
g(x) := − \frac{d^2}{dx^2}u(x).
\end{equation*}
Prove that

  1.  $u \in C^2([0,1])$.
  2.  $u$ satisfies the boundary conditions.
  3.  The parameters
  $$\sigma_1(x) = 1, x \in (0,1)$$
  and
  $$\sigma_2(x) =
  \begin{cases}
  1, & x \in (\tfrac{1}{4}, \tfrac{3}{4}) \\
  (x - \tfrac{1}{4})^2 + 1, & x \in (0, \tfrac{1}{4}] \\
  (x - \tfrac{3}{4})^2 + 1, & x \in [\tfrac{3}{4}, 1)
  \end{cases}
  $$
  both satisfy the equation (1).